### Instalação das dependências

Este notebook requer autenticação ativa no Google Cloud Platform. Execute gcloud auth application-default login antes de iniciar ou utilize uma conta de serviço com as permissões corretas.

In [ ]:
!pip install google-cloud-aiplatform -q

# Código 2.4 — Implantação Conceitual no Google Cloud Vertex AI

**Objetivo:** Demonstrar os passos programmaticos para registrar um modelo treinado no **Vertex AI Model Registry** e criar um **Endpoint** de inferência na nuvem.

### Passo 1: Inicialização do SDK do Google Cloud Vertex AI

Configuramos as credenciais e parâmetros do projeto GCP.

In [ ]:
from google.cloud import aiplatform

# Parâmetros de configuração da nuvem
PROJECT_ID = "meu-projeto-gcp-123"
LOCATION = "us-central1"
STAGING_BUCKET = "gs://meu-bucket-artefatos-ml"

# Inicializar SDK
aiplatform.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET
)
print("SDK do Vertex AI inicializado com sucesso!")

**Entendendo as variáveis de conexão com o GCP:**
- `PROJECT_ID`: Identificador único do projeto no Google Cloud Platform.
- `LOCATION`: Região do datacenter GCP onde os recursos de IA serão provisionados (ex: `us-central1`).
- `STAGING_BUCKET`: Endereço do Bucket no Cloud Storage usado para armazenar temporariamente os artefatos serializados.

### Passo 2: Registro do Modelo no Vertex AI Model Registry

Enviamos a imagem Docker e os artefatos para o catálogo central de modelos.

In [ ]:
# Upload do modelo para o Registry
model = aiplatform.Model.upload(
    display_name="modelo-predicao-falha-v1",
    serving_container_image_uri="us-docker.pkg.dev/cloudrun/container/hello",
    serving_container_predict_route="/predict",
    serving_container_health_route="/health",
)

print(f"Modelo registrado no Vertex AI. Resource Name: {model.resource_name}")

**Entendendo os parâmetros de registro do modelo:**
- `display_name`: Nome amigável do modelo exibido no painel do Vertex AI.
- `serving_container_image_uri`: URI do container Docker pré-construído armazenado no Artifact Registry.
- `serving_container_predict_route`: Rota HTTP do container responsável por receber os dados e retornar predições.
- `serving_container_health_route`: Rota HTTP do container utilizada pelo GCP para verificar a disponibilidade do serviço.

### Passo 3: Criação do Endpoint e Deploy do Modelo

Alocamos infraestrutura computacional gerenciada (máquina virtual) para servir predições.

In [ ]:
# Criar Endpoint gerenciado
endpoint = model.deploy(
    machine_type="n1-standard-2",
    min_replica_count=1,
    max_replica_count=3,
    accelerator_count=0,
)

print(f"Endpoint ativo e pronto para receber chamadas: {endpoint.resource_name}")

**Entendendo os parâmetros de infraestrutura de implantação:**
- `machine_type="n1-standard-2"`: Tipo da máquina virtual alocada (2 vCPUs e 7,5 GB de RAM).
- `min_replica_count=1`: Número mínimo de instâncias ativas prontas para responder requisições.
- `max_replica_count=3`: Limite máximo de instâncias para auto-escalonamento em picos de tráfego.
- `accelerator_count=0`: Indica que a inferência utilizará apenas CPU, sem a necessidade de placas de vídeo (GPUs).

---

## Conclusão da Aula — Vertex AI / Cloud Deploy

Implantar modelos gerenciados em plataformas Cloud como o **Vertex AI** abstrai o gerenciamento de servidor e permite escalar a API de inferência de acordo com a demanda de acessos.